# Understanding How PyTorch Computes Gradients

**What you'll learn in this notebook:**
- What derivatives are and why they matter for machine learning
- How `requires_grad` works — leaf tensors vs intermediate tensors
- How PyTorch builds a computation graph during the forward pass
- How `backward()` traverses the graph to compute gradients
- Manual gradient verification — compute by hand, check with autograd
- When and why to use `torch.no_grad()` and `torch.inference_mode()`
- The gradient accumulation pitfall (and how to fix it)
- Writing custom autograd functions with forward/backward
- Higher-order gradients, Jacobians, and Hessians

**Prerequisites:** Notebook 01 (Tensors). Basic calculus (derivatives, chain rule).  
**Time:** ~50 minutes  
**Hardware:** CPU only — no GPU required!

In [ ]:
import torch
import torch.nn as nn
print(f"PyTorch version: {torch.__version__}")

## 1. Quick Derivative Refresher

A **derivative** tells you the rate of change. For `f(x) = x^2`:
- `f'(x) = 2x` — at `x=3`, the slope is `6`
- This means: if you nudge x by a tiny amount, the output changes by ~6x that amount

In machine learning, we compute derivatives of a **loss function** with respect to **model parameters** to know which direction to update them.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

y.backward()

print(f"f(x) = x^2")
print(f"x = {x.item()}")
print(f"f(x) = {y.item()}")
print(f"f'(x) = 2x = {x.grad.item()}")
print(f"Expected: 2 * 3 = 6.0  ✓" if x.grad.item() == 6.0 else "Mismatch!")

## 2. requires_grad — Tracking Computations

When you set `requires_grad=True` on a tensor, PyTorch starts recording all operations performed on it. This builds a **computation graph** that can be traversed backward to compute gradients.

**Leaf tensors** — tensors you create directly (e.g., model parameters)  
**Non-leaf tensors** — tensors created by operations (e.g., `y = x * 2`)

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b        # non-leaf: created by multiplication
d = c + a         # non-leaf: created by addition

print(f"a: leaf={a.is_leaf}, requires_grad={a.requires_grad}, grad_fn={a.grad_fn}")
print(f"b: leaf={b.is_leaf}, requires_grad={b.requires_grad}, grad_fn={b.grad_fn}")
print(f"c: leaf={c.is_leaf}, requires_grad={c.requires_grad}, grad_fn={c.grad_fn}")
print(f"d: leaf={d.is_leaf}, requires_grad={d.requires_grad}, grad_fn={d.grad_fn}")

no_grad_tensor = torch.tensor(5.0)
print(f"\nDefault: requires_grad={no_grad_tensor.requires_grad} (no tracking)")

## 3. The Computation Graph

Every operation creates a node in the graph. The `grad_fn` attribute tells you which operation created the tensor. Let's trace a chain:

```
x ──→ [multiply by w] ──→ z ──→ [add b] ──→ y ──→ [square] ──→ loss
```

In [ ]:
x = torch.tensor(2.0)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

z = x * w       # z = 2 * 3 = 6
y = z + b        # y = 6 + 1 = 7
loss = y ** 2    # loss = 49

print("=== Forward pass ===")
print(f"z = x*w = {z.item()}")
print(f"y = z+b = {y.item()}")
print(f"loss = y^2 = {loss.item()}")

print("\n=== Computation graph (grad_fn chain) ===")
print(f"loss.grad_fn: {loss.grad_fn}")
print(f"  └─ y.grad_fn: {y.grad_fn}")
print(f"       └─ z.grad_fn: {z.grad_fn}")

loss.backward()
print(f"\n=== Gradients (chain rule) ===")
print(f"dloss/dw = dloss/dy * dy/dz * dz/dw = 2y * 1 * x = 2*7*2 = {w.grad.item()}")
print(f"dloss/db = dloss/dy * dy/db          = 2y * 1     = 2*7   = {b.grad.item()}")

## 4. Manual Gradient Verification

Let's compute gradients by hand for a more complex function and verify with autograd.

`f(x, y) = sin(x) * y^2 + exp(y)`

Partial derivatives:
- `df/dx = cos(x) * y^2`
- `df/dy = 2*y*sin(x) + exp(y)`

In [ ]:
import math

x = torch.tensor(math.pi / 4, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)

f = torch.sin(x) * y**2 + torch.exp(y)
f.backward()

manual_df_dx = math.cos(math.pi / 4) * 4.0
manual_df_dy = 2 * 2.0 * math.sin(math.pi / 4) + math.exp(2.0)

print(f"f(pi/4, 2) = sin(pi/4)*4 + e^2 = {f.item():.6f}")
print(f"\ndf/dx:")
print(f"  autograd: {x.grad.item():.6f}")
print(f"  manual:   {manual_df_dx:.6f}")
print(f"  match: {torch.isclose(x.grad, torch.tensor(manual_df_dx))}")
print(f"\ndf/dy:")
print(f"  autograd: {y.grad.item():.6f}")
print(f"  manual:   {manual_df_dy:.6f}")
print(f"  match: {torch.isclose(y.grad, torch.tensor(manual_df_dy))}")

## 5. Numerical Gradient Check

You can verify any gradient numerically using finite differences:  
`f'(x) ≈ (f(x+h) - f(x-h)) / (2h)` for small h.

In [ ]:
def f(x):
    return torch.sin(x) * x**2

x = torch.tensor(1.5, requires_grad=True)
y = f(x)
y.backward()
autograd_result = x.grad.item()

h = 1e-5
numerical_result = (f(torch.tensor(1.5 + h)).item() - f(torch.tensor(1.5 - h)).item()) / (2 * h)

print(f"f(x) = sin(x) * x^2 at x=1.5")
print(f"Autograd gradient:  {autograd_result:.8f}")
print(f"Numerical gradient: {numerical_result:.8f}")
print(f"Absolute error:     {abs(autograd_result - numerical_result):.2e}")

## 6. torch.no_grad() and torch.inference_mode()

During inference, you don't need gradients. Disabling them saves memory and speeds up computation.

| Context Manager | Gradient Tracking | Allows in-place on views | Use case |
|----------------|:-:|:-:|---------|
| Default | Yes | Yes | Training |
| `torch.no_grad()` | No | Yes | Validation, manual param updates |
| `torch.inference_mode()` | No | No (stricter) | Pure inference (fastest) |

In [ ]:
x = torch.randn(3, requires_grad=True)

print("=== Default: grad tracking ON ===")
y = x * 2
print(f"y.requires_grad: {y.requires_grad}, y.grad_fn: {y.grad_fn}")

print("\n=== torch.no_grad(): grad tracking OFF ===")
with torch.no_grad():
    y = x * 2
    print(f"y.requires_grad: {y.requires_grad}, y.grad_fn: {y.grad_fn}")

print("\n=== torch.inference_mode(): strictest ===")
with torch.inference_mode():
    y = x * 2
    print(f"y.requires_grad: {y.requires_grad}")

print("\n--- When to use each ---")
print("no_grad():       validation loop, manual weight updates")
print("inference_mode(): production inference, benchmarking")

## 7. The Gradient Accumulation Pitfall

This is the **#1 autograd bug** for beginners. By default, PyTorch **accumulates** gradients — calling `backward()` multiple times adds to `.grad` instead of replacing it.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)

print("=== THE BUG: forgetting to zero gradients ===")
for i in range(4):
    loss = (w * 3) ** 2  # dloss/dw = 2 * (w*3) * 3 = 36 at w=2
    loss.backward()
    print(f"  Step {i}: loss={loss.item():.1f}, w.grad={w.grad.item():.1f} (should be 36.0!)")

print("\n=== THE FIX: zero_grad before backward ===")
w = torch.tensor(2.0, requires_grad=True)
for i in range(4):
    if w.grad is not None:
        w.grad.zero_()    # ← The fix!
    loss = (w * 3) ** 2
    loss.backward()
    print(f"  Step {i}: loss={loss.item():.1f}, w.grad={w.grad.item():.1f}")

## 8. Gradients with Vectors

When the output is not a scalar, you need to specify a `gradient` argument (or reduce to scalar first). In practice, loss functions always produce a scalar, so this rarely comes up.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2  # y = [1, 4, 9]

# Option 1: reduce to scalar first (most common)
y.sum().backward()
print(f"d(sum(x^2))/dx = 2x = {x.grad}")

# Option 2: provide gradient argument (for vector-Jacobian product)
x2 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y2 = x2 ** 2
v = torch.tensor([1.0, 1.0, 1.0])  # this is like weighting each output equally
y2.backward(gradient=v)
print(f"VJP with v=ones: {x2.grad}")  # same result

## 9. Custom Autograd Function

Sometimes you need to define the forward AND backward pass yourself. Use `torch.autograd.Function` for this.

Let's implement **ReLU** from scratch: `relu(x) = max(0, x)` with gradient `1 if x > 0 else 0`.

In [ ]:
class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return input.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output * (input > 0).float()
        return grad_input

my_relu = MyReLU.apply

x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
y = my_relu(x)
y.sum().backward()

print(f"Input:    {x.data}")
print(f"MyReLU:   {y.data}")
print(f"Gradient: {x.grad}")
print(f"\nVerify against torch.relu:")
x2 = x.detach().clone().requires_grad_(True)
torch.relu(x2).sum().backward()
print(f"torch.relu grad: {x2.grad}")
print(f"Match: {torch.allclose(x.grad, x2.grad)}")

## 10. Higher-Order Gradients

PyTorch can compute gradients of gradients. This is useful for:
- Hessian computation (second derivatives)
- Meta-learning (MAML)
- Physics-informed neural networks

Use `create_graph=True` in backward to keep the graph for further differentiation.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3  # y = x^3, y' = 3x^2, y'' = 6x

# First derivative
dy_dx = torch.autograd.grad(y, x, create_graph=True)[0]
print(f"f(x) = x^3 at x=2")
print(f"f'(x) = 3x^2 = {dy_dx.item()} (expected: {3 * 4})")

# Second derivative
d2y_dx2 = torch.autograd.grad(dy_dx, x, create_graph=True)[0]
print(f"f''(x) = 6x = {d2y_dx2.item()} (expected: {6 * 2})")

# Third derivative
d3y_dx3 = torch.autograd.grad(d2y_dx2, x)[0]
print(f"f'''(x) = 6 = {d3y_dx3.item()} (expected: 6)")

## 11. Jacobian and Hessian

The **Jacobian** is the matrix of all partial derivatives for a vector-valued function.  
The **Hessian** is the matrix of second partial derivatives for a scalar-valued function.

PyTorch provides `torch.autograd.functional.jacobian` and `hessian` for these.

In [ ]:
from torch.autograd.functional import jacobian, hessian

# Jacobian: f(x) = [x0^2 + x1, x0 * x1]
def f_vec(x):
    return torch.stack([x[0]**2 + x[1], x[0] * x[1]])

x = torch.tensor([2.0, 3.0])
J = jacobian(f_vec, x)
print(f"f([2,3]) = {f_vec(x)}")
print(f"Jacobian:\n{J}")
print(f"  Expected: [[2*x0, 1], [x1, x0]] = [[4, 1], [3, 2]]")

# Hessian: g(x) = x0^2 * x1 + x1^3
def g(x):
    return x[0]**2 * x[1] + x[1]**3

H = hessian(g, x)
print(f"\ng([2,3]) = {g(x)}")
print(f"Hessian:\n{H}")
print(f"  Expected: [[2*x1, 2*x0], [2*x0, 6*x1]] = [[6, 4], [4, 18]]")

## 12. Detaching from the Graph

Sometimes you need to "detach" a tensor from the computation graph — treating it as a constant.

Common uses:
- Freezing part of a network during training
- Target networks in reinforcement learning
- Avoiding backprop through certain operations

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
z = y.detach()  # z has the same value but no grad connection

print(f"y = x^2 = {y.item()}, requires_grad={y.requires_grad}")
print(f"z = y.detach() = {z.item()}, requires_grad={z.requires_grad}")

# z acts as a constant — gradient doesn't flow through it
w = z * x  # dw/dx = z (treating z as constant) = 9
w.backward()
print(f"\nw = z * x where z = detach(x^2)")
print(f"dw/dx = z = {x.grad.item()}")
print(f"(NOT 3*x^2=27, because z is treated as constant 9)")

## 🧪 Exercise: Gradient Descent from Scratch

**Task:** Implement gradient descent to find the minimum of:

`f(x, y) = (x - 3)^2 + (y + 1)^2`

The minimum is at `(3, -1)` where `f = 0`. Can your optimizer find it?

In [ ]:
torch.manual_seed(42)
x = torch.randn(1, requires_grad=True)
y = torch.randn(1, requires_grad=True)
lr = 0.1

print(f"Starting point: x={x.item():.4f}, y={y.item():.4f}")
print(f"Target minimum: x=3.0000, y=-1.0000\n")

for step in range(50):
    # Forward: compute loss
    loss = (x - 3)**2 + (y + 1)**2

    # Backward: compute gradients
    loss.backward()

    # Update: gradient descent step (no_grad to avoid tracking this)
    with torch.no_grad():
        x -= lr * x.grad
        y -= lr * y.grad

    # Zero gradients for next iteration
    x.grad.zero_()
    y.grad.zero_()

    if step % 10 == 0 or step == 49:
        print(f"Step {step:3d}: x={x.item():7.4f}, y={y.item():7.4f}, loss={loss.item():.6f}")

print(f"\nFinal: x={x.item():.4f}, y={y.item():.4f}")
print(f"Target: x=3.0000, y=-1.0000")

### Try It Yourself

1. Change the learning rate — what happens with `lr=0.01`? `lr=1.0`? `lr=2.0`?
2. Try optimizing `f(x,y) = x^4 + y^4 - 2*x^2 - 2*y^2` (has 4 minima!)
3. Use `torch.optim.SGD` instead of manual updates — compare the code.

In [ ]:
# Try it yourself! Experiment with the optimizer here:

# Using torch.optim.SGD for comparison
x = torch.tensor([0.0], requires_grad=True)
y = torch.tensor([0.0], requires_grad=True)
optimizer = torch.optim.SGD([x, y], lr=0.1)

for step in range(50):
    optimizer.zero_grad()
    loss = (x - 3)**2 + (y + 1)**2
    loss.backward()
    optimizer.step()

print(f"With torch.optim.SGD: x={x.item():.4f}, y={y.item():.4f}")

## Key Takeaways

1. **`requires_grad=True`** tells PyTorch to track operations for gradient computation
2. **The computation graph** is built dynamically during the forward pass (define-by-run)
3. **`backward()`** traverses the graph using the chain rule to compute gradients
4. **Always `zero_grad()`** before each backward pass to avoid gradient accumulation
5. **`torch.no_grad()`** for validation/updates; **`inference_mode()`** for pure inference
6. **`detach()`** breaks gradient flow — useful for freezing parts of computation
7. **Custom autograd functions** let you define your own forward/backward
8. **Higher-order gradients** work via `create_graph=True`
9. **Numerical gradient checking** is your debugging friend: `(f(x+h) - f(x-h)) / 2h`

**Next up:** Notebook 03 — Building Neural Networks